In [1]:
import pandas as pd
import os
import ast
import re

In [7]:
list_name = os.listdir('./')
study_group = 'QUE'

In [8]:
list_name

['demo_script.ipynb',
 'QUE_SDD',
 'Q_nhanes_codebook_ACQ_B_resultados.csv',
 'Q_nhanes_codebook_ACQ_C_resultados.csv',
 'Q_nhanes_codebook_ACQ_D_resultados.csv',
 'Q_nhanes_codebook_ACQ_E_resultados.csv',
 'Q_nhanes_codebook_ACQ_F_resultados.csv',
 'Q_nhanes_codebook_ACQ_G_resultados.csv',
 'Q_nhanes_codebook_ACQ_H_resultados.csv',
 'Q_nhanes_codebook_ACQ_I_resultados.csv',
 'Q_nhanes_codebook_ACQ_J_resultados.csv',
 'Q_nhanes_codebook_ACQ_L_resultados.csv',
 'Q_nhanes_codebook_ACQ_resultados.csv',
 'Q_nhanes_codebook_AGQ_D_resultados.csv',
 'Q_nhanes_codebook_ALQY_F_resultados.csv',
 'Q_nhanes_codebook_ALQ_B_resultados.csv',
 'Q_nhanes_codebook_ALQ_C_resultados.csv',
 'Q_nhanes_codebook_ALQ_D_resultados.csv',
 'Q_nhanes_codebook_ALQ_E_resultados.csv',
 'Q_nhanes_codebook_ALQ_F_resultados.csv',
 'Q_nhanes_codebook_ALQ_G_resultados.csv',
 'Q_nhanes_codebook_ALQ_H_resultados.csv',
 'Q_nhanes_codebook_ALQ_I_resultados.csv',
 'Q_nhanes_codebook_ALQ_J_resultados.csv',
 'Q_nhanes_codebook_A

In [9]:

name_tables = [palavra.split("Q_nhanes_codebook_")[1] for palavra in list_name if palavra.endswith('.csv')]
name_tables = [palavra.split("_") for palavra in name_tables if palavra.endswith('resultados.csv')]

nome_unique = []
for lista in name_tables:
    if len(lista[0])>1:
        nome_unique.append(lista[0])
    elif len(lista[0])==1:
        nome_unique.append(lista[1])
        
nome_unique = list(set(nome_unique))

In [10]:
def process_string_to_dict(string):
    string = string.split('; Count')[0]
    # Substituir os `;` por vírgulas e remover espaços extras
    formatted_string = re.sub(r";", ",", string)
    formatted_string = re.sub(r"=", ":", formatted_string)
    # Adicionar chaves externas para transformá-la em um dicionário
    formatted_string = "{" + formatted_string + "}"
    # Substituir chaves como "Code.or.Value" por "Code" e "Value.Description" por "Description"
    formatted_string = re.sub(r"Code\.or\.Value", '"Code"', formatted_string)
    formatted_string = re.sub(r"Value\.Description", '"Description"', formatted_string)

    return eval(formatted_string)

In [12]:
for nome in nome_unique:
    lista_filtrada = [item for item in list_name if nome in item]
    # print()
    # print("********************************")
    # print(lista_filtrada)
    
    df_now_total = pd.DataFrame()
    for item in lista_filtrada:
        df_now = pd.read_csv(item)
        df_now_total = pd.concat([df_now_total, df_now])
        
        
    list_Variable_Name = df_now_total['Variable_Name'].unique()
    df_codebook_final = pd.DataFrame()
    for variable in list_Variable_Name:
        df_now = df_now_total[df_now_total['Variable_Name']==variable]
        list_english = list(df_now['English_Text'].unique())
        list_Target = list(df_now['Target'].unique())
        list_English_Instructions = list(df_now['English_Instructions'].unique())
        df_now['English_Text'] = str(list_english)
        df_now['Target'] = str(list_Target)
        df_now['English_Instructions'] = str(list_English_Instructions)
        df_codebook_final = pd.concat([df_codebook_final, df_now])
    
    #display(df_codebook_final)

    rows = []
    for _, row in df_codebook_final.iterrows():
        # Ignorar linhas onde Tabela_Valores é NaN
        if pd.isna(row['Tabela_Valores']):
            rows.append({
                    "Column": row["Variable_Name"],
                    "Code": None,
                    "Label": None,
                    "English_Text": row["English_Text"],
                    "Target": row["Target"],
                    "English_Instructions": row["English_Instructions"]
                })
            continue
        
        # Extrair os dados de Code e Value.Description
        valores = row['Tabela_Valores']
        
        try:
            # Usando ast.literal_eval para transformar a string em dicionário
            parsed_values = process_string_to_dict(valores)
            codes = parsed_values["Code"]
            codes = [str(item).title() for item in codes]
            descriptions = parsed_values["Description"]
            descriptions = [str(item).title() for item in descriptions]
            # Criar uma nova linha para cada combinação de Code e Description
            for code, description in zip(codes, descriptions):
                rows.append({
                    "Column": row["Variable_Name"],
                    "Code": code,
                    "Label": description,
                    "English_Text": row["English_Text"],
                    "Target": row["Target"],
                    "English_Instructions": row["English_Instructions"]
                })
        except Exception as e:
            print(f"Erro ao processar a linha: {row['Tabela_Valores']}")
            print(e)

    # Criar um novo DataFrame com os dados processados
    df_expanded = pd.DataFrame(rows)
    df_expanded['Class'] = None
    df_expanded['Other For'] = None
    
    df_expanded.drop_duplicates(inplace=True)
    df_expanded.reset_index(inplace=True, drop=True)
    
    df_expanded.to_csv(f'./QUE_SDD/{study_group}_{nome}_codebook.csv', sep=',', encoding='utf-8')
    
    dictionary_mapping = df_now_total[['Variable_Name']]
    dictionary_mapping = dictionary_mapping.drop_duplicates(subset='Variable_Name', keep='first')
    dictionary_mapping['Attribute'] = None
    dictionary_mapping['attributeOf'] = None
    dictionary_mapping['Unit'] = None
    dictionary_mapping['Time'] = None
    dictionary_mapping['Entity'] = None
    dictionary_mapping['Role'] = None
    dictionary_mapping['Relation'] = None
    dictionary_mapping['inRelationTo'] = None
    dictionary_mapping['wasDerivedFrom'] = None
    dictionary_mapping['wasGeneratedBy'] = None
    dictionary_mapping.reset_index(inplace=True, drop=True)
    dictionary_mapping.rename(columns={"Variable_Name": "Column"}, inplace=True)
    dictionary_mapping.reset_index(inplace=True, drop=True)
    dictionary_mapping.to_csv(f'./QUE_SDD/{study_group}_{nome}_dictionary_mapping.csv', sep=',', encoding='utf-8')
    
    
    
    
    # display(df_expanded)
    # display(dictionary_mapping)
    # break

C:\Users\l-oen\AppData\Local\Temp\ipykernel_11392\3868593426.py:20: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_now['English_Text'] = str(list_english)
C:\Users\l-oen\AppData\Local\Temp\ipykernel_11392\3868593426.py:21: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_now['Target'] = str(list_Target)
C:\Users\l-oen\AppData\Local\Temp\ipykernel_11392\3868593426.py:22: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = val

Erro ao processar a linha: Code.or.Value=["1", "2", "."]; Value.Description=["Sample Person Interview Questionnaire Targets (B(2-11) 
      and (B(16-150)", "MEC CAPI Questionnaire Targets (B(12-15)", "Missing"]; Count=["8749", "610", "0"]; Cumulative=["8749", "9359", "9359"]; Skip.to.Item=["NA", "NA", "NA"]
unterminated string literal (detected at line 1) (<string>, line 1)
Erro ao processar a linha: Code.or.Value=["1", "2", "."]; Value.Description=["Sample Person Interview Questionnaire Targets (B(2-11) 
      and (B(16-150)", "MEC CAPI Questionnaire Targets (B(12-15)", "Missing"]; Count=["9119", "652", "0"]; Cumulative=["9119", "9771", "9771"]; Skip.to.Item=["NA", "NA", "NA"]
unterminated string literal (detected at line 1) (<string>, line 1)
Erro ao processar a linha: Code.or.Value=["1", "2", "."]; Value.Description=["Sample Person Interview Questionnaire Targets (B(2-11) 
      and (B(16-150))", "MEC CAPI Questionnaire Targets (B(12-15))", "Missing"]; Count=["8477", "630", "0"]; C

C:\Users\l-oen\AppData\Local\Temp\ipykernel_11392\3868593426.py:20: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_now['English_Text'] = str(list_english)
C:\Users\l-oen\AppData\Local\Temp\ipykernel_11392\3868593426.py:21: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_now['Target'] = str(list_Target)
C:\Users\l-oen\AppData\Local\Temp\ipykernel_11392\3868593426.py:22: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = val

Erro ao processar a linha: Code.or.Value=["1", "2", "3", "4", "7", "9", "."]; Value.Description=["little or no psoriasis,", "only a few patches (that could be covered 
by one or two palms of {your/his/her} 
hand),", "scattered patches (that could be covered 
between three and ten palms of {your/ 
his/her} hand), or", "extensive psoriasis (covering large areas of 
the body, that would be more than ten 
palms of {your/his/her} hand)?", "Refused", "Don't know", "Missing"]; Count=["92", "39", "25", "9", "0", "2", "9197"]; Cumulative=["92", "131", "156", "165", "165", "167", "9364"]; Skip.to.Item=["NA", "NA", "NA", "NA", "NA", "NA", "NA"]
unterminated string literal (detected at line 1) (<string>, line 1)
Erro ao processar a linha: Code.or.Value=["1", "2", "3", "4", "7", "9", "."]; Value.Description=["little or no psoriasis,", "only a few patches (that could be covered 
by one or two palms of {your/his/her} 
hand),", "scattered patches (that could be covered 
between three and ten palms of 

C:\Users\l-oen\AppData\Local\Temp\ipykernel_11392\3868593426.py:20: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_now['English_Text'] = str(list_english)
C:\Users\l-oen\AppData\Local\Temp\ipykernel_11392\3868593426.py:21: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_now['Target'] = str(list_Target)
C:\Users\l-oen\AppData\Local\Temp\ipykernel_11392\3868593426.py:22: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = val

Erro ao processar a linha: Code.or.Value=["11", "."]; Value.Description=["2% fat milk (includes "low fat milk" not further specified),", "Missing"]; Count=["190", "10849"]; Cumulative=["190", "11039"]; Skip.to.Item=["NA", "NA"]
invalid syntax. Perhaps you forgot a comma? (<string>, line 1)
Erro ao processar a linha: Code.or.Value=["11", "."]; Value.Description=["2% fat milk (includes "low fat milk" not further specified),", "Missing"]; Count=["3000", "8039"]; Cumulative=["3000", "11039"]; Skip.to.Item=["NA", "NA"]
invalid syntax. Perhaps you forgot a comma? (<string>, line 1)
Erro ao processar a linha: Code.or.Value=["11", "."]; Value.Description=["2% fat milk (includes "low fat milk" not further specified),", "Missing"]; Count=["208", "9914"]; Cumulative=["208", "10122"]; Skip.to.Item=["NA", "NA"]
invalid syntax. Perhaps you forgot a comma? (<string>, line 1)
Erro ao processar a linha: Code.or.Value=["11", "."]; Value.Description=["2% fat milk (includes "low fat milk" not further spec

C:\Users\l-oen\AppData\Local\Temp\ipykernel_11392\3868593426.py:20: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_now['English_Text'] = str(list_english)
C:\Users\l-oen\AppData\Local\Temp\ipykernel_11392\3868593426.py:21: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_now['Target'] = str(list_Target)
C:\Users\l-oen\AppData\Local\Temp\ipykernel_11392\3868593426.py:22: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = val

Erro ao processar a linha: Code.or.Value=["1", "2", "3", "4", "5", "7", "9", "."]; Value.Description=["Once per month", "2-3 times per month", "4-8 times per month (about 1-2 times per 
  week)", "9-24 times per month (about 3-6 times per 
  week)", "25-30 times per month (one or more times 
  per day)", "Refused", "Don't know", "Missing"]; Count=["80", "124", "211", "246", "289", "0", "2", "4350"]; Cumulative=["80", "204", "415", "661", "950", "950", "952", "5302"]; Skip.to.Item=["NA", "NA", "NA", "NA", "NA", "NA", "NA", "NA"]
unterminated string literal (detected at line 1) (<string>, line 1)
Erro ao processar a linha: Code.or.Value=["1", "2", "3", "4", "5", "7", "9", "."]; Value.Description=["Once per month", "2-3 times per month", "4-8 times per month (about 1-2 times per 
  week)", "9-24 times per month (about 3-6 times per 
  week)", "25-30 times per month (one or more times 
  per day)", "Refused", "Don't know", "Missing"]; Count=["72", "146", "211", "191", "245", "2", "1", "392

C:\Users\l-oen\AppData\Local\Temp\ipykernel_11392\3868593426.py:20: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_now['English_Text'] = str(list_english)
C:\Users\l-oen\AppData\Local\Temp\ipykernel_11392\3868593426.py:21: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_now['Target'] = str(list_Target)
C:\Users\l-oen\AppData\Local\Temp\ipykernel_11392\3868593426.py:22: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = val